In [6]:
# ============================================================
# 1  PIPELINE CONFIGURATION
# ============================================================
# Local files expected in the SAME folder as this notebook:
#   - labs_clean_source_copy.parquet     (blood tests)
#   - 01_clinical.parquet                (admissions / clinical covariates)
#   - new_df_episodekey_deathdate.csv    (death-date lookup)

from pathlib import Path
import pandas as pd
import numpy as np
import os
import warnings

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)

# --- Folder = same folder as the notebook ---
BASE_DIR = Path.cwd()

# --- Local data paths ---
LABS_PATH       = BASE_DIR / 'labs_clean_source_copy.parquet'
ADMISSIONS_PATH = BASE_DIR / '01_clinical.parquet'
DEATH_PATH      = BASE_DIR / 'new_df_episodekey_deathdate.csv'

# --- Temporal filter (None = no filter) ---
DATE_START = '2024-02-01'   # Inclusive (>=)
DATE_END   = '2026-01-31'   # Inclusive (<=)

# --- Columns to load from the labs parquet ---
COLS_TO_KEEP = [
    'Subject_identifier',
    'Date',
    'Method_text',
    'Measure',
    'Numeric_value',
    'Range_string',   # <-- clinical reference range (LLN/ULN) per measurement
]

# --- Admissions column mapping ---
COL_PATIENT_ID_ADMISSIONS = 'CloudPatientId'
COL_START_DATE_ADMISSIONS = 'startDate'
COL_END_DATE_ADMISSIONS   = 'endDate'

# --- Measures column mapping ---
COL_PATIENT_ID_MEASURES = 'Subject_identifier'
COL_DATE_MEASURES       = 'Date'
COL_TEST_NAME           = 'Method_text'
COL_MEASURE_UNIT        = 'Measure'
COL_VALUE               = 'Numeric_value'

# --- Death-data column mapping ---
COL_DEATH_PATIENT_ID = 'CloudPatientId'   # join key with admissions
COL_DEATH_DATE       = 'dateOfDeath'

# --- Blood-test whitelist ----------------------------------------------------
# Only these analyses are kept; everything else is dropped immediately
# after loading the parquet, so we never spend time cleaning unused tests.
BLOOD_TESTS_TO_KEEP = [
    'EMOCROMO - Ematocrito',
    'EMOCROMO - Emoglobina',
    'EMOCROMO - Globuli bianchi',
    'EMOCROMO - Globuli rossi',
    'EMOCROMO - MCH (Emoglobina Corpuscolare Media)',
    'EMOCROMO - MCHC (Conc.Emoglobinica Corpusc.Media)',
    'EMOCROMO - MCV (Volume Globulare Medio)',
    'EMOCROMO - RDW (Ind.Distribuzione Eritrocitaria)',
    'FORMULA  - Linfociti',
    'FORMULA  - Eosinofili',
    'FORMULA  - Monociti',
    'FORMULA  - Neutrofili',
    'PIASTRINE - MPV (Volume Piastrinico Medio)',
    'PIASTRINE - Piastrine',
    'S-ALANINA AMINOTRANSFERASI',
    'S-ASPARTATO AMINOTRANSFERASI',
    'S-BILIRUBINA TOTALE',
    'S-CREATININA',
    'S-LATTICODEIDROGENASI',
    'S-POTASSIO',
    'S-PROTEINA C REATTIVA',
    'S-SODIO',
    'S-UREA',
]


In [7]:
# ============================================================
#  2  DATA LOADING (local files)
# ============================================================

# --- Load blood-test parquet (only the columns we need) ---
df_measures_raw = pd.read_parquet(LABS_PATH, columns=COLS_TO_KEEP)

# --- Restrict to whitelisted blood tests ONLY -------------------------------
n_before = len(df_measures_raw)
df_measures_raw = df_measures_raw[
    df_measures_raw[COL_TEST_NAME].isin(BLOOD_TESTS_TO_KEEP)
].copy()
print(f"Blood tests filtered to whitelist: {n_before:,} -> {len(df_measures_raw):,} rows "
      f"({df_measures_raw[COL_TEST_NAME].nunique()} distinct tests kept "
      f"out of {len(BLOOD_TESTS_TO_KEEP)} expected)")

# Sanity check: do we actually find every whitelisted test in the data?
found = set(df_measures_raw[COL_TEST_NAME].unique())
missing = [t for t in BLOOD_TESTS_TO_KEEP if t not in found]
if missing:
    print(f"  WARNING: these whitelisted tests were not found in the parquet:")
    for t in missing:
        print(f"   - {t!r}")

# --- Load admissions / clinical parquet ---
df_admissions_raw = pd.read_parquet(ADMISSIONS_PATH)

# --- Load death-date CSV ---
df_death_raw = pd.read_csv(DEATH_PATH)

# Sanity check death file structure: we MUST have CloudPatientId + dateOfDeath.
# If they're missing it usually means a different separator, BOM, or capitalisation.
print(f"\ndf_death_raw columns: {list(df_death_raw.columns)}")
for col in [COL_DEATH_PATIENT_ID, COL_DEATH_DATE]:
    if col not in df_death_raw.columns:
        raise KeyError(
            f"Column {col!r} not found in {DEATH_PATH.name}. "
            f"Available columns: {list(df_death_raw.columns)}"
        )

print(f"\nBlood tests loaded:  {df_measures_raw.shape}")
print(f"Admissions loaded:   {df_admissions_raw.shape}")
print(f"Death lookup loaded: {df_death_raw.shape}")


Blood tests filtered to whitelist: 3,028,013 -> 1,335,409 rows (23 distinct tests kept out of 23 expected)

df_death_raw columns: ['pubID', 'CloudPatientId', 'endDate', 'episode_key', 'Alive_at_4_May_2026', 'dateOfDeath']

Blood tests loaded:  (1335409, 6)
Admissions loaded:   (1352, 28)
Death lookup loaded: (1249, 6)


In [8]:
print(df_admissions_raw['hospitalWard'].value_counts(dropna=False))


hospitalWard
A    734
B    618
Name: count, dtype: Int64


In [9]:
# ============================================================
#  2.bis  ADMISSION COVARIATE COLUMNS
# ============================================================
# These columns describe the patient at admission and will be
# replicated across every longitudinal datapoint for that patient.
#
# NOTE: CloudPatientId is the patient ID (handled separately as the join key).
# NOTE: startDate / endDate are kept BOTH as the admission window (used for
#       filtering and for the 'day' computation) AND as final covariates.
# NOTE: dateOfDeath is added later from new_df_episodekey_deathdate.csv.

ADMISSION_COVARIATE_COLS = [
    'startDate',
    'endDate',
    'sex',
    'hospitalWard',
    'age',
    'pksYear',
    'alcYears',
    'LOS_days',
    'daysInER',
    'daysInER_capped',
    'ER_last_day',
    'ward_startDate',
    'admLast12m',
    'admLast6m',
    'admECOG',
    'admBarthelScore',
    'admBradenScore',
    'cirsTotalScore',
    'admReason',
    'dischargedAlive',
    'dischargeMode',
    'solidTumor',
    'solidTumor_cat',
]

# Death-related columns merged in later (only dateOfDeath, no Alive flag)
DEATH_COVARIATE_COLS = ['dateOfDeath']


In [10]:
print("n unique patients: ", df_admissions_raw['CloudPatientId'].nunique())
print("n admissions per patient: ", df_admissions_raw.groupby('CloudPatientId').size().value_counts())

n unique patients:  1249
n admissions per patient:  1    1161
2      76
3      11
6       1
Name: count, dtype: int64


In [11]:
print("min date: ", df_admissions_raw[['startDate','endDate']].min())
print("max date: ", df_admissions_raw[['startDate','endDate']].max())

min date:  startDate   2023-10-21
endDate     2024-02-03
dtype: datetime64[ns]
max date:  startDate   2026-01-20
endDate     2026-01-30
dtype: datetime64[ns]


In [12]:
print("min date: ", df_measures_raw['Date'].min())

min date:  2008-01-12T09:11:00+00:00


In [9]:
# ============================================================
#  3  CLEANING AND TYPE CONVERSION
# ============================================================

df_measures   = df_measures_raw.copy()
df_admissions = df_admissions_raw.copy()

# 1. Convert measure dates (ISO timestamp -> date)
df_measures[COL_DATE_MEASURES] = pd.to_datetime(
    df_measures[COL_DATE_MEASURES], utc=True
).dt.date
df_measures = df_measures[df_measures[COL_DATE_MEASURES] >= pd.to_datetime(DATE_START).date()]

# 2. Convert admission dates
df_admissions[COL_START_DATE_ADMISSIONS] = pd.to_datetime(
    df_admissions[COL_START_DATE_ADMISSIONS]
).dt.date

df_admissions[COL_END_DATE_ADMISSIONS] = pd.to_datetime(
    df_admissions[COL_END_DATE_ADMISSIONS]
).dt.date

# 3. Normalise patient IDs (lowercase for consistent matching)
df_measures[COL_PATIENT_ID_MEASURES] = (
    df_measures[COL_PATIENT_ID_MEASURES].str.strip().str.lower()
)
df_admissions[COL_PATIENT_ID_ADMISSIONS] = (
    df_admissions[COL_PATIENT_ID_ADMISSIONS].str.strip().str.lower()
)

# 4. Drop rows without a numeric value
df_measures = df_measures.dropna(subset=[COL_VALUE])

print(f"  Measures after cleaning: {df_measures.shape}")
print(f"   Unique patients: {df_measures[COL_PATIENT_ID_MEASURES].nunique()}")
print(f"   Unique tests:    {df_measures[COL_TEST_NAME].nunique()}")
print(f"   Date range:      {df_measures[COL_DATE_MEASURES].min()} -> "
      f"{df_measures[COL_DATE_MEASURES].max()}")


  Measures after cleaning: (971118, 6)
   Unique patients: 1641
   Unique tests:    23
   Date range:      2024-02-01 -> 2026-05-04


In [5]:
# ============================================================
#  4  MEASUREMENT-UNIT CHECK PER TEST (REPORT ONLY)
# ============================================================

units_per_test = (
    df_measures
    .groupby(COL_TEST_NAME)[COL_MEASURE_UNIT]
    .nunique()
    .reset_index()
    .rename(columns={COL_MEASURE_UNIT: 'n_units'})
)

problematic_tests = units_per_test[units_per_test['n_units'] > 1]

if len(problematic_tests) > 0:
    print(f"INFO  {len(problematic_tests)} tests have MULTIPLE measurement units:")
    print("=" * 60)
    for _, row in problematic_tests.iterrows():
        test_name = row[COL_TEST_NAME]
        counts = (
            df_measures[df_measures[COL_TEST_NAME] == test_name][COL_MEASURE_UNIT]
            .value_counts()
        )
        print(f"\n  >> {test_name}:")
        for unit, count in counts.items():
            print(f"     - '{unit}': {count} measurements")

    print("\n" + "=" * 60)
    print("   Rules for handling these tests are defined in the next section.")
    print("   No data has been modified or removed in this section.")
else:
    print("OK  All tests have a single measurement unit. No action required.")

print(f"\n  Current state:")
print(f"   Total measurements:          {len(df_measures):,}")
print(f"   Unique tests:                {df_measures[COL_TEST_NAME].nunique()}")
print(f"   Tests with multiple units:   {len(problematic_tests)}")


INFO  7 tests have MULTIPLE measurement units:

  >> EMOCROMO - Emoglobina:
     - 'g/L': 31269 measurements
     - 'g/dL': 10978 measurements

  >> EMOCROMO - MCHC (Conc.Emoglobinica Corpusc.Media):
     - 'g/L': 31266 measurements
     - 'g Hb /dL': 10974 measurements

  >> FORMULA  - Eosinofili:
     - '10^9/L': 38457 measurements
     - '%': 38457 measurements

  >> FORMULA  - Linfociti:
     - '10^9/L': 38457 measurements
     - '%': 38457 measurements

  >> FORMULA  - Monociti:
     - '%': 38456 measurements
     - '10^9/L': 38456 measurements

  >> FORMULA  - Neutrofili:
     - '%': 38458 measurements
     - '10^9/L': 38458 measurements

  >> S-CREATININA:
     - 'mg/dL': 35534 measurements
     - 'µmol/L': 3 measurements

   Rules for handling these tests are defined in the next section.
   No data has been modified or removed in this section.

  Current state:
   Total measurements:          971,118
   Unique tests:                23
   Tests with multiple units:   7


In [6]:
# ============================================================
#  5  RULES FOR TESTS WITH MULTIPLE MEASUREMENT UNITS
# ============================================================
# Limited to the 23 whitelisted analyses.
# Tests not listed here either:
#   - are already on a single unit in the source data, or
#   - are not part of the whitelist (they were filtered out at load time).

UNIT_RULES = {
    # --- Leukocyte differential: keep absolute values (10^9/L) ---
    'FORMULA  - Eosinofili':  {'keep_unit': '10^9/L'},
    'FORMULA  - Linfociti':   {'keep_unit': '10^9/L'},
    'FORMULA  - Monociti':    {'keep_unit': '10^9/L'},
    'FORMULA  - Neutrofili':  {'keep_unit': '10^9/L'},

    # --- Hemoglobin: keep g/dL (convert g/L -> g/dL by /10) ---
    'EMOCROMO - Emoglobina': {
        'keep_unit': 'g/dL',
        'convert': [{
            'from_unit': 'g/L',
            'to_unit':   'g/dL',
            'func': lambda x: x / 10,
        }],
    },

    # --- MCHC: keep g Hb /dL (convert g/L -> g Hb /dL by /10) ---
    'EMOCROMO - MCHC (Conc.Emoglobinica Corpusc.Media)': {
        'keep_unit': 'g Hb /dL',
        'convert': [{
            'from_unit': 'g/L',
            'to_unit':   'g Hb /dL',
            'func': lambda x: x / 10,
        }],
    },

    # --- Creatinine: keep mg/dL (convert µmol/L -> mg/dL) ---
    'S-CREATININA': {
        'keep_unit': 'mg/dL',
        'convert': [{
            'from_unit': 'µmol/L',
            'to_unit':   'mg/dL',
            'func': lambda x: x * 0.0113,
        }],
    },
}


In [7]:
# ============================================================
#  5.1  APPLY MEASUREMENT-UNIT RULES
# ============================================================

n_before = len(df_measures)
tests_dropped       = []
tests_converted      = []
tests_filtered_unit  = []

for test_name, rules in UNIT_RULES.items():
    mask_test = df_measures[COL_TEST_NAME] == test_name

    if mask_test.sum() == 0:
        continue

    # CASE 1: drop the entire test
    if rules.get('drop_all', False):
        n_drop = mask_test.sum()
        df_measures = df_measures[~mask_test]
        tests_dropped.append(f"  DROPPED  {test_name}: {n_drop} measurements removed")
        continue

    # CASE 2: conversions
    if 'convert' in rules:
        for conv in rules['convert']:
            mask_convert = mask_test & (df_measures[COL_MEASURE_UNIT] == conv['from_unit'])
            n_convert = mask_convert.sum()
            if n_convert > 0:
                df_measures.loc[mask_convert, COL_VALUE] = (
                    df_measures.loc[mask_convert, COL_VALUE].apply(conv['func'])
                )
                df_measures.loc[mask_convert, COL_MEASURE_UNIT] = conv['to_unit']
                tests_converted.append(
                    f"  CONVERTED  {test_name}: {n_convert} values "
                    f"from '{conv['from_unit']}' -> '{conv['to_unit']}'"
                )

    # CASE 3: keep only the specified unit (after possible conversions)
    keep_unit = rules['keep_unit']
    mask_test = df_measures[COL_TEST_NAME] == test_name
    mask_wrong_unit = mask_test & (df_measures[COL_MEASURE_UNIT] != keep_unit)
    n_wrong = mask_wrong_unit.sum()
    if n_wrong > 0:
        df_measures = df_measures[~mask_wrong_unit]
        tests_filtered_unit.append(
            f"  FILTERED  {test_name}: {n_wrong} measurements with unit "
            f"!= '{keep_unit}' removed"
        )

# --- Report ---
n_after = len(df_measures)
print("=" * 70)
print("  MEASUREMENT-UNIT CLEANUP REPORT")
print("=" * 70)

if tests_dropped:
    print(f"\n  TESTS DROPPED ENTIRELY ({len(tests_dropped)}):")
    for msg in tests_dropped:
        print(msg)

if tests_converted:
    print(f"\n  CONVERSIONS APPLIED ({len(tests_converted)}):")
    for msg in tests_converted:
        print(msg)

if tests_filtered_unit:
    print(f"\n  UNITS FILTERED ({len(tests_filtered_unit)}):")
    for msg in tests_filtered_unit:
        print(msg)

print(f"\n{'=' * 70}")
print(f"  Measurements: {n_before:,} -> {n_after:,} (removed: {n_before - n_after:,})")
print(f"  Unique tests remaining: {df_measures[COL_TEST_NAME].nunique()}")

# --- Final verification ---
units_check = (
    df_measures.groupby(COL_TEST_NAME)[COL_MEASURE_UNIT].nunique()
)
still_problematic = units_check[units_check > 1]

if len(still_problematic) > 0:
    print(f"\n  WARNING: {len(still_problematic)} tests still have multiple units:")
    for test_name in still_problematic.index:
        units = (
            df_measures[df_measures[COL_TEST_NAME] == test_name][COL_MEASURE_UNIT]
            .value_counts()
        )
        print(f"   - {test_name}: {dict(units)}")
    print("   Add these tests to UNIT_RULES!")
else:
    print(f"\n  OK: all tests now have a single measurement unit!")



  MEASUREMENT-UNIT CLEANUP REPORT

  CONVERSIONS APPLIED (3):
  CONVERTED  EMOCROMO - Emoglobina: 31269 values from 'g/L' -> 'g/dL'
  CONVERTED  EMOCROMO - MCHC (Conc.Emoglobinica Corpusc.Media): 31266 values from 'g/L' -> 'g Hb /dL'
  CONVERTED  S-CREATININA: 3 values from 'µmol/L' -> 'mg/dL'

  UNITS FILTERED (4):
  FILTERED  FORMULA  - Eosinofili: 38457 measurements with unit != '10^9/L' removed
  FILTERED  FORMULA  - Linfociti: 38457 measurements with unit != '10^9/L' removed
  FILTERED  FORMULA  - Monociti: 38456 measurements with unit != '10^9/L' removed
  FILTERED  FORMULA  - Neutrofili: 38458 measurements with unit != '10^9/L' removed

  Measurements: 971,118 -> 817,290 (removed: 153,828)
  Unique tests remaining: 23

  OK: all tests now have a single measurement unit!


In [8]:
# ============================================================
#  5.2  EXTRACT CLINICAL REFERENCE RANGES (LLN/ULN) FROM Range_string
# ============================================================
# Parse Range_string into (LLN, ULN) and run a CONSISTENCY CHECK per test.
# Where clinicians entered several different ranges we:
#   1. drop the ones in the WRONG unit/scale (per-test overrides below),
#   2. take the count-WEIGHTED AVERAGE of the kept ranges (each measurement
#      is one row, so the plain row-mean of lln/uln IS the count-weighting).
# Keyed by the pre-rename wide column name "<Method_text> (<unit>)"; remapped
# to short display names in the final save cell.

import re

COL_RANGE = 'Range_string'
if COL_RANGE not in df_measures.columns:
    raise KeyError(
        f"{COL_RANGE!r} not found in df_measures. Add it to COLS_TO_KEEP "
        f"(section 1) and re-run from the top."
    )

def parse_reference_range(s):
    """Parse a lab reference-range string -> (lln, uln) as floats.
    np.nan marks an absent/open bound. Handles 'a - b', 'a-b', comma
    decimals ('13,0-17,0'), open bounds ('< 5', 'fino a 5', '> 60'),
    en/em dashes. 0 or >2 numbers -> (nan, nan) so the consistency
    check flags it as unparsed rather than guessing."""
    if s is None or (isinstance(s, float) and np.isnan(s)):
        return (np.nan, np.nan)
    t = str(s).strip().lower()
    if t in ('', 'nan', 'none', '-', 'n.d.', 'nd', '/'):
        return (np.nan, np.nan)
    t = t.replace('\u2013', '-').replace('\u2014', '-')      # en/em dash -> hyphen
    t = re.sub(r'(?<=\d),(?=\d)', '.', t)                    # decimal comma -> dot
    nums = [float(x) for x in re.findall(r'\d+(?:\.\d+)?', t)]
    has_lt = ('<' in t) or ('\u2264' in t) or ('fino a' in t)
    has_gt = ('>' in t) or ('\u2265' in t)
    if len(nums) == 2:
        lo, hi = sorted(nums)
        return (lo, hi)
    if len(nums) == 1:
        if has_lt:
            return (0.0, nums[0])        # e.g. CRP "< 5"
        if has_gt:
            return (nums[0], np.nan)     # e.g. "> 60"
        return (np.nan, np.nan)
    return (np.nan, np.nan)

# ── Per-test overrides for INCONSISTENT ranges ───────────────────────────────
# For a listed test, keep ONLY the (LLN, ULN) pairs given here (wrong-unit /
# wrong-scale variants are dropped); the final range is the count-weighted
# average of the kept rows. Tests NOT listed keep every parsed range and are
# weighted-averaged as-is (e.g. sex-specific ranges collapse to a population
# band — acceptable since the model is not sex-stratified).
# Decisions below reflect manual clinical review of the consistency report.
REF_RANGE_KEEP = {
    # Creatinine: keep the mg/dL ranges; drop the µmol/L ones (44-110 / 44-97).
    'S-CREATININA (mg/dL)':
        [(0.5, 1.25), (0.5, 1.1)],
    # Hemoglobin: keep the g/dL ranges; drop the g/L ones (140-180 / 120-160).
    'EMOCROMO - Emoglobina (g/dL)':
        [(14.0, 18.0), (12.0, 16.0)],
    # MCHC: same interval, different scale — the study uses g Hb /dL (31-36).
    'EMOCROMO - MCHC (Conc.Emoglobinica Corpusc.Media) (g Hb /dL)':
        [(31.0, 36.0)],
}
KEEP_TOL = 1e-6

def _row_kept(tcol, lln, uln):
    keep = REF_RANGE_KEEP.get(tcol)
    if keep is None:
        return not (np.isnan(lln) and np.isnan(uln))    # default: keep any parsed row
    for (kl, ku) in keep:
        ok_l = np.isclose(lln, kl, atol=KEEP_TOL) if not np.isnan(lln) else np.isnan(kl)
        ok_u = np.isclose(uln, ku, atol=KEEP_TOL) if not np.isnan(uln) else np.isnan(ku)
        if ok_l and ok_u:
            return True
    return False

work = df_measures[[COL_TEST_NAME, COL_MEASURE_UNIT, COL_RANGE]].copy()
work['test_col_name'] = (
    work[COL_TEST_NAME].astype(str) + ' (' + work[COL_MEASURE_UNIT].astype(str) + ')'
)
_parsed = work[COL_RANGE].apply(parse_reference_range)
work['lln'] = _parsed.apply(lambda p: p[0])
work['uln'] = _parsed.apply(lambda p: p[1])

print("=" * 80)
print("  REFERENCE-RANGE CONSISTENCY CHECK  (weighted-average policy)")
print("=" * 80)

ref_rows = []
for tcol, g in work.groupby('test_col_name'):
    both_nan = g[['lln', 'uln']].isna().all(axis=1)
    n_unparsed = int(both_nan.sum())
    parsed = g.loc[~both_nan]
    parsed_counts = (
        parsed.groupby(['lln', 'uln'], dropna=False).size().sort_values(ascending=False)
    )
    n_distinct = parsed_counts.shape[0]

    kept_mask = parsed.apply(lambda r: _row_kept(tcol, r['lln'], r['uln']), axis=1)
    kept = parsed.loc[kept_mask] if len(parsed) else parsed

    flag = '' if n_distinct <= 1 else '   <-- INCONSISTENT'
    print(f"\n  {tcol}{flag}")
    print(f"    rows: {len(g):,} | distinct parsed: {n_distinct} | "
          f"unparsed: {n_unparsed} | kept rows: {len(kept):,}")
    for (lo, hi), c in parsed_counts.items():
        mark = 'keep' if _row_kept(tcol, lo, hi) else 'DROP'
        print(f"      [{lo}, {hi}]  x{c:,}   [{mark}]")
    if n_unparsed:
        ex = g.loc[both_nan, COL_RANGE].dropna().astype(str).unique()[:3]
        if len(ex):
            print(f"    e.g. unparsed strings: {list(ex)}")

    if len(kept) == 0:
        print("    WARNING: no rows kept for this test — check REF_RANGE_KEEP. Skipped.")
        continue

    lln_w = float(kept['lln'].mean())        # count-weighted (row-mean)
    uln_w = float(kept['uln'].mean())
    if (n_distinct > 1) or (len(kept) < len(parsed)):
        print(f"    -> weighted range: [{lln_w:.4g}, {uln_w:.4g}]")

    ref_rows.append({
        'test_col_name':       tcol,
        'lln':                 round(lln_w, 6),
        'uln':                 round(uln_w, 6),
        'n_distinct_ranges':   int(n_distinct),
        'n_ranges_kept':       int(kept.groupby(['lln', 'uln']).ngroups),
        'n_measurements':      int(len(g)),
        'n_measurements_kept': int(len(kept)),
        'n_unparsed':          n_unparsed,
    })

ref_ranges_longname = pd.DataFrame(ref_rows)
n_inconsistent = int((ref_ranges_longname['n_distinct_ranges'] > 1).sum()) if len(ref_ranges_longname) else 0
print("\n" + "=" * 80)
print(f"  Reference ranges for {len(ref_ranges_longname)} tests "
      f"({n_inconsistent} were inconsistent — resolved by drop-then-weighted-average).")
print("=" * 80)


  REFERENCE-RANGE CONSISTENCY CHECK  (weighted-average policy)

  EMOCROMO - Ematocrito (%)   <-- INCONSISTENT
    rows: 42,247 | distinct parsed: 2 | unparsed: 0 | kept rows: 42,247
      [42.0, 52.0]  x25,422   [keep]
      [37.0, 47.0]  x16,825   [keep]
    -> weighted range: [40.01, 50.01]

  EMOCROMO - Emoglobina (g/dL)   <-- INCONSISTENT
    rows: 42,247 | distinct parsed: 4 | unparsed: 0 | kept rows: 10,978
      [140.0, 180.0]  x19,151   [DROP]
      [120.0, 160.0]  x12,118   [DROP]
      [14.0, 18.0]  x6,271   [keep]
      [12.0, 16.0]  x4,707   [keep]
    -> weighted range: [13.14, 17.14]

  EMOCROMO - Globuli bianchi (10^9/L)
    rows: 42,256 | distinct parsed: 1 | unparsed: 0 | kept rows: 42,256
      [4.8, 10.8]  x42,256   [keep]

  EMOCROMO - Globuli rossi (10^12/L)   <-- INCONSISTENT
    rows: 42,252 | distinct parsed: 2 | unparsed: 0 | kept rows: 42,252
      [4.7, 6.1]  x25,426   [keep]
      [4.2, 5.4]  x16,826   [keep]
    -> weighted range: [4.501, 5.821]

  EMOCROM

In [9]:
# ============================================================
#  6  SELECT PATIENTS WITH A COMPLETE ADMISSION IN THE DATE WINDOW
#     THEN KEEP THE LAST ADMISSION PER PATIENT
#     THEN MERGE dateOfDeath FROM new_df_episodekey_deathdate.csv
# ============================================================

date_start = pd.to_datetime(DATE_START).date()
date_end   = pd.to_datetime(DATE_END).date()

# Step 1 — keep only admissions whose full stay falls within [DATE_START, DATE_END]
mask_complete = (
    (df_admissions[COL_START_DATE_ADMISSIONS] >= date_start) &
    (df_admissions[COL_END_DATE_ADMISSIONS]   <= date_end)
)
df_admissions_window = df_admissions[mask_complete].copy()

print(f"  Admissions with complete stay in [{DATE_START}, {DATE_END}]: "
      f"{len(df_admissions_window):,}  "
      f"({df_admissions_window[COL_PATIENT_ID_ADMISSIONS].nunique()} unique patients)")

# Step 2 — for each patient keep the LAST complete admission (latest endDate)
df_admissions_last = (
    df_admissions_window
    .sort_values(COL_END_DATE_ADMISSIONS)
    .groupby(COL_PATIENT_ID_ADMISSIONS)
    .last()
    .reset_index()
)

# Step 2.bis — DROP the dateOfDeath column from clinical, we'll bring our own
# from the CSV. Avoids dateOfDeath_x / dateOfDeath_y at merge time.
if COL_DEATH_DATE in df_admissions_last.columns:
    print(f"  INFO: dropping pre-existing {COL_DEATH_DATE!r} column from clinical "
          f"data (we'll use the CSV's version instead).")
    df_admissions_last = df_admissions_last.drop(columns=[COL_DEATH_DATE])

# Step 3 — MERGE dateOfDeath from the death-lookup file
df_death = df_death_raw.copy()
for col in [COL_DEATH_PATIENT_ID, COL_DEATH_DATE]:
    if col not in df_death.columns:
        raise KeyError(
            f"Column {col!r} not found in df_death_raw. "
            f"Available: {list(df_death.columns)}"
        )

# Normalise patient id to lowercase to match the admissions df
df_death[COL_DEATH_PATIENT_ID] = (
    df_death[COL_DEATH_PATIENT_ID].astype(str).str.strip().str.lower()
)
df_death[COL_DEATH_DATE] = pd.to_datetime(
    df_death[COL_DEATH_DATE], errors='coerce'
).dt.date

df_death_slim = (
    df_death[[COL_DEATH_PATIENT_ID, COL_DEATH_DATE]]
    .drop_duplicates(subset=[COL_DEATH_PATIENT_ID])
)

df_admissions_last = df_admissions_last.merge(
    df_death_slim,
    on=COL_DEATH_PATIENT_ID,
    how='left',
)

# Sanity check: no _x / _y suffixes
suffix_clash = [c for c in df_admissions_last.columns if c.endswith('_x') or c.endswith('_y')]
if suffix_clash:
    raise RuntimeError(
        f"Unexpected _x/_y columns after merge: {suffix_clash}. "
        f"Some other column collided between admissions and death CSV."
    )

n_with_death = df_admissions_last[COL_DEATH_DATE].notna().sum()
n_without    = df_admissions_last[COL_DEATH_DATE].isna().sum()
print(f"  After merging death-lookup: {n_with_death} patients have dateOfDeath, "
      f"{n_without} do not.")

# Step 4 — build the final column list (ID + base dates + covariates + death)
# startDate / endDate appear BOTH in admissions_base_cols and in
# ADMISSION_COVARIATE_COLS. We dedupe with dict.fromkeys (preserves order).
admissions_base_cols = [
    COL_PATIENT_ID_ADMISSIONS,
    COL_START_DATE_ADMISSIONS,
    COL_END_DATE_ADMISSIONS,
]

available_covariates = [
    c for c in ADMISSION_COVARIATE_COLS if c in df_admissions_last.columns
]
missing_covariates = [
    c for c in ADMISSION_COVARIATE_COLS if c not in df_admissions_last.columns
]
available_death_cols = [
    c for c in DEATH_COVARIATE_COLS if c in df_admissions_last.columns
]

if missing_covariates:
    print(f"  WARNING: covariate columns not found in admissions dataset "
          f"(will be skipped): {missing_covariates}")

# All covariates that will be propagated to the longitudinal dataset
available_covariates = available_covariates + available_death_cols

# Dedupe (startDate/endDate are in both base and covariates) — preserve order
admissions_cols_to_keep = list(dict.fromkeys(admissions_base_cols + available_covariates))
df_admissions_last = df_admissions_last[admissions_cols_to_keep]

print(f"  Last complete admission per patient: {df_admissions_last.shape}")
print(f"   Unique patients:           {df_admissions_last[COL_PATIENT_ID_ADMISSIONS].nunique()}")
print(f"   Admission window:          {df_admissions_last[COL_START_DATE_ADMISSIONS].min()} "
      f"-> {df_admissions_last[COL_END_DATE_ADMISSIONS].max()}")
print(f"   Covariate columns kept:    {available_covariates}")

assert df_admissions_last[COL_PATIENT_ID_ADMISSIONS].is_unique, \
    "ERROR: duplicate patients after last-admission selection!"
print("  OK  Each patient has exactly 1 admission record.")


  Admissions with complete stay in [2024-02-01, 2026-01-31]: 1,306  (1206 unique patients)
  INFO: dropping pre-existing 'dateOfDeath' column from clinical data (we'll use the CSV's version instead).
  After merging death-lookup: 483 patients have dateOfDeath, 723 do not.
  Last complete admission per patient: (1206, 25)
   Unique patients:           1206
   Admission window:          2024-02-02 -> 2026-01-30
   Covariate columns kept:    ['startDate', 'endDate', 'sex', 'hospitalWard', 'age', 'pksYear', 'alcYears', 'LOS_days', 'daysInER', 'daysInER_capped', 'ER_last_day', 'ward_startDate', 'admLast12m', 'admLast6m', 'admECOG', 'admBarthelScore', 'admBradenScore', 'cirsTotalScore', 'admReason', 'dischargedAlive', 'dischargeMode', 'solidTumor', 'solidTumor_cat', 'dateOfDeath']
  OK  Each patient has exactly 1 admission record.


In [10]:
# ============================================================
#  7  MERGE MEASURES + ADMISSIONS -> FILTER TO ADMISSION WINDOW
# ============================================================
# At this point df_admissions_last contains only patients whose
# last complete admission falls entirely within [DATE_START, DATE_END].
# We now join their blood-test records and keep only tests taken
# DURING that admission.

df_merged = df_measures.merge(
    df_admissions_last,
    left_on=COL_PATIENT_ID_MEASURES,
    right_on=COL_PATIENT_ID_ADMISSIONS,
    how='inner',
)

print(f"  Measures before merge:                          {len(df_measures):,}")
print(f"  Measures after merge (patients with admission): {len(df_merged):,}")

# Keep only tests performed DURING the admission
df_filtered = df_merged[
    (df_merged[COL_DATE_MEASURES] >= df_merged[COL_START_DATE_ADMISSIONS]) &
    (df_merged[COL_DATE_MEASURES] <= df_merged[COL_END_DATE_ADMISSIONS])
].copy()

print(f"  Measures within admission window:               {len(df_filtered):,}")
print(f"   Patients with at least one test during admission: "
      f"{df_filtered[COL_PATIENT_ID_MEASURES].nunique()}")
print(f"   Unique tests present: {df_filtered[COL_TEST_NAME].nunique()}")


  Measures before merge:                          817,290
  Measures after merge (patients with admission): 644,120
  Measures within admission window:               296,029
   Patients with at least one test during admission: 1203
   Unique tests present: 23


In [11]:
# ============================================================
#  8  DAY SINCE ADMISSION + PIVOT + COVARIATES
# ============================================================

# --- 1. Compute 'day' = days since admission start ---
# Convert to datetime for subtraction, then to integer days
df_filtered['day'] = (
    pd.to_datetime(df_filtered[COL_DATE_MEASURES])
    - pd.to_datetime(df_filtered[COL_START_DATE_ADMISSIONS])
).dt.days

print(f"  OK  'day' column computed (0 = admission date)")
print(f"   Max day across all patients: {df_filtered['day'].max()}")
print(f"   Day distribution per patient:")
print(df_filtered.groupby(COL_PATIENT_ID_MEASURES)['day'].max().describe())

# --- 2. Build column names with measurement unit ---
# e.g. "FORMULA - Eosinofili (10^9/L)"
df_filtered['test_col_name'] = df_filtered.apply(
    lambda row: f"{row[COL_TEST_NAME]} ({row[COL_MEASURE_UNIT]})", axis=1
)

# --- 3. Pivot: one row per (patient, date, day) ---
df_longitudinal = df_filtered.pivot_table(
    index=[COL_PATIENT_ID_MEASURES, COL_DATE_MEASURES, 'day'],
    columns='test_col_name',
    values=COL_VALUE,
    aggfunc='mean',       # average if duplicates on the same day/test
).reset_index()

# --- 4. Attach patient-level covariates ---
# These are static (one value per patient) so we merge them in
if available_covariates:
    covariate_df = df_admissions_last[
        [COL_PATIENT_ID_ADMISSIONS] + available_covariates
    ].copy()

    df_longitudinal = df_longitudinal.merge(
        covariate_df,
        left_on=COL_PATIENT_ID_MEASURES,
        right_on=COL_PATIENT_ID_ADMISSIONS,
        how='left',
    )

    # Drop the duplicate ID column if it was created
    if (COL_PATIENT_ID_ADMISSIONS != COL_PATIENT_ID_MEASURES
            and COL_PATIENT_ID_ADMISSIONS in df_longitudinal.columns):
        df_longitudinal.drop(columns=[COL_PATIENT_ID_ADMISSIONS], inplace=True)

# --- 5. Rename & sort ---
df_longitudinal = df_longitudinal.rename(columns={
    COL_PATIENT_ID_MEASURES: 'patient_id',
    COL_DATE_MEASURES:       'analysis_date',
})

df_longitudinal = df_longitudinal.sort_values(
    ['patient_id', 'day']
).reset_index(drop=True)

# Count test columns (everything except structural + covariates)
structural_cols = ['patient_id', 'analysis_date', 'day'] + available_covariates
test_columns = [c for c in df_longitudinal.columns if c not in structural_cols]

print(f"\n  OK  FINAL LONGITUDINAL DATASET:")
print(f"   Shape:             {df_longitudinal.shape}")
print(f"   Patients:          {df_longitudinal['patient_id'].nunique()}")
print(f"   Test columns:      {len(test_columns)}")
print(f"   Covariate columns: {available_covariates}")
print(f"\n   First rows:")
df_longitudinal.head(20)


  OK  'day' column computed (0 = admission date)
   Max day across all patients: 396
   Day distribution per patient:


count    1203.000000
mean       19.339152
std        18.035868
min         0.000000
25%        10.000000
50%        15.000000
75%        24.000000
max       396.000000
Name: day, dtype: float64


In [12]:
# ============================================================
#  9  NaN DISTRIBUTION ANALYSIS AND THRESHOLD FILTER
# ============================================================

# --- CONFIGURATION ---
NAN_THRESHOLD = 0.705  # Drop columns with more than this fraction of NaN
                       # e.g. 0.3 = 30%, 0.7 = 70%
                       # None = report only, do not remove anything

# --- Structural columns ---
STRUCTURAL_COLS_FINAL = ['patient_id', 'analysis_date', 'day']
non_structural_cols = [c for c in df_longitudinal.columns if c not in STRUCTURAL_COLS_FINAL]

# 1. Compute NaN % per test column
nan_stats = (
    df_longitudinal[non_structural_cols]
    .isnull()
    .mean()
    .sort_values(ascending=False)
    .reset_index()
)
nan_stats.columns = ['test', 'nan_pct']
nan_stats['nan_count']   = df_longitudinal[nan_stats['test']].isnull().sum().values
nan_stats['valid_count'] = len(df_longitudinal) - nan_stats['nan_count']

# 2. Full report
print("=" * 80)
print("  NaN DISTRIBUTION PER COLUMN (TESTS)")
print("=" * 80)
print(f"\n   Total rows in dataset: {len(df_longitudinal)}")
print(f"   Total test columns:    {len(non_structural_cols)}")

# Completeness bands
bins   = [0, 0.1, 0.2, 0.3, 0.5, 0.7, 0.9, 1.0]
labels = ['0-10%', '10-20%', '20-30%', '30-50%', '50-70%', '70-90%', '90-100%']
nan_stats['nan_band'] = pd.cut(
    nan_stats['nan_pct'], bins=bins, labels=labels, include_lowest=True
)

print(f"\n  Column distribution by NaN band:")
print("-" * 45)
band_counts = nan_stats['nan_band'].value_counts().sort_index()
for band, count in band_counts.items():
    bar = '#' * count
    print(f"   {band:>10s} NaN: {count:>4d} columns  {bar}")

# 3. Detail for all columns
print(f"\n{'─' * 80}")
print(f"{'Test':<55s} {'NaN %':>7s} {'NaN':>6s} {'Valid':>7s}")
print(f"{'─' * 80}")
for _, row in nan_stats.iterrows():
    marker = ' DROP' if NAN_THRESHOLD and row['nan_pct'] > NAN_THRESHOLD else ' KEEP'
    print(
        f"   {row['test']:<52s} "
        f"{row['nan_pct']:>6.1%} "
        f"{int(row['nan_count']):>6d} "
        f"{int(row['valid_count']):>7d}"
        f"{marker}"
    )

# 4. Apply threshold (if configured)
if NAN_THRESHOLD is not None:
    cols_to_drop_nan = nan_stats[nan_stats['nan_pct'] > NAN_THRESHOLD]['test'].tolist()
    cols_to_keep_nan = nan_stats[nan_stats['nan_pct'] <= NAN_THRESHOLD]['test'].tolist()

    print(f"\n{'=' * 80}")
    print(f"  THRESHOLD SET AT: {NAN_THRESHOLD:.0%}")
    print(f"   KEEP:   {len(cols_to_keep_nan)} columns")
    print(f"   DROP:   {len(cols_to_drop_nan)} columns")

    if cols_to_drop_nan:
        print(f"\n   Columns removed:")
        for col in cols_to_drop_nan:
            pct = nan_stats[nan_stats['test'] == col]['nan_pct'].values[0]
            print(f"      DROP  {col} ({pct:.1%} NaN)")

        df_longitudinal = df_longitudinal.drop(columns=cols_to_drop_nan)

        print(f"\n   Shape after removal: {df_longitudinal.shape}")
        print(f"      (before: {len(STRUCTURAL_COLS_FINAL) + len(non_structural_cols)} columns -> "
              f"now: {df_longitudinal.shape[1]} columns)")
    else:
        print(f"\n   OK  No column exceeds the threshold. Nothing to remove!")
else:
    print(f"\n{'=' * 80}")
    print("  INFO  NAN_THRESHOLD = None -> Report only, no columns removed.")
    print("   To activate filtering, set NAN_THRESHOLD to a value (e.g. 0.5)")

  NaN DISTRIBUTION PER COLUMN (TESTS)

   Total rows in dataset: 14760
   Total test columns:    47

  Column distribution by NaN band:
---------------------------------------------
        0-10% NaN:   30 columns  ##############################
       10-20% NaN:    9 columns  #########
       20-30% NaN:    0 columns  
       30-50% NaN:    4 columns  ####
       50-70% NaN:    4 columns  ####
       70-90% NaN:    0 columns  
      90-100% NaN:    0 columns  

────────────────────────────────────────────────────────────────────────────────
Test                                                      NaN %    NaN   Valid
────────────────────────────────────────────────────────────────────────────────
   S-LATTICODEIDROGENASI (U/L)                           68.8%  10161    4599 KEEP
   S-BILIRUBINA TOTALE (mg/dL)                           56.5%   8344    6416 KEEP
   dateOfDeath                                           56.3%   8312    6448 KEEP
   alcYears                               

In [13]:
df_longitudinal.head(20)


,patient_id,analysis_date,day,EMOCROMO - Ematocrito (%),EMOCROMO - Emoglobina (g/dL),EMOCROMO - Globuli bianchi (10^9/L),EMOCROMO - Globuli rossi (10^12/L),EMOCROMO - MCH (Emoglobina Corpuscolare Media) (pg/Cell),EMOCROMO - MCHC (Conc.Emoglobinica Corpusc.Media) (g Hb /dL),EMOCROMO - MCV (Volume Globulare Medio) (fL),EMOCROMO - RDW (Ind.Distribuzione Eritrocitaria) (CV %),FORMULA - Eosinofili (10^9/L),FORMULA - Linfociti (10^9/L),FORMULA - Monociti (10^9/L),FORMULA - Neutrofili (10^9/L),PIASTRINE - MPV (Volume Piastrinico Medio) (fL),PIASTRINE - Piastrine (10^9/L),S-ALANINA AMINOTRANSFERASI (U/L),S-ASPARTATO AMINOTRANSFERASI (U/L),S-BILIRUBINA TOTALE (mg/dL),S-CREATININA (mg/dL),S-LATTICODEIDROGENASI (U/L),S-POTASSIO (mmol/L),S-PROTEINA C REATTIVA (mg/L),S-SODIO (mmol/L),S-UREA (mg/dL),startDate,endDate,sex,hospitalWard,age,pksYear,alcYears,LOS_days,daysInER,daysInER_capped,ER_last_day,ward_startDate,admLast12m,admLast6m,admECOG,admBarthelScore,admBradenScore,cirsTotalScore,admReason,dischargedAlive,dischargeMode,solidTumor,solidTumor_cat,dateOfDeath
0,0032c17f-9c2b-861a-1346-3a1d24b3e281,2024-04-17,0,38.299999,12.0,9.5,3.94,30.500000,31.299999,97.199997,14.4,0.3,1.8,0.6,6.8,10.1,246.0,24.0,30.0,NaN,2.37,248.0,4.520,84.699997,151.800003,117.0,2024-04-17,2024-05-19,m,B,89.0,0.0,NaN,33,NaN,NaN,NaT,NaT,0.0,0.0,4.0,55.0,19.0,7,2 | 4,0,NaN,0.0,No cancer,2024-05-19
1,0032c17f-9c2b-861a-1346-3a1d24b3e281,2024-04-18,1,41.099998,12.8,13.5,4.22,30.299999,31.100000,97.400002,14.6,0.1,1.7,0.8,10.8,9.8,249.0,25.0,29.0,NaN,2.33,NaN,4.160,57.099998,154.899994,110.0,2024-04-17,2024-05-19,m,B,89.0,0.0,NaN,33,NaN,NaN,NaT,NaT,0.0,0.0,4.0,55.0,19.0,7,2 | 4,0,NaN,0.0,No cancer,2024-05-19
2,0032c17f-9c2b-861a-1346-3a1d24b3e281,2024-04-19,2,38.400002,12.1,10.4,3.96,30.600000,31.500000,97.000000,14.8,0.2,1.5,0.7,7.9,10.0,245.0,NaN,NaN,NaN,2.60,NaN,NaN,71.300003,155.100006,NaN,2024-04-17,2024-05-19,m,B,89.0,0.0,NaN,33,NaN,NaN,NaT,NaT,0.0,0.0,4.0,55.0,19.0,7,2 | 4,0,NaN,0.0,No cancer,2024-05-19
3,0032c17f-9c2b-861a-1346-3a1d24b3e281,2024-04-20,3,38.299999,11.9,10.3,3.93,30.299999,31.100000,97.500000,14.6,0.3,1.9,0.7,7.3,10.4,237.0,NaN,NaN,NaN,2.10,NaN,NaN,NaN,150.399994,NaN,2024-04-17,2024-05-19,m,B,89.0,0.0,NaN,33,NaN,NaN,NaT,NaT,0.0,0.0,4.0,55.0,19.0,7,2 | 4,0,NaN,0.0,No cancer,2024-05-19
4,0032c17f-9c2b-861a-1346-3a1d24b3e281,2024-04-22,5,35.799999,11.4,9.8,3.77,30.200001,31.799999,95.000000,14.3,0.2,2.0,0.9,6.6,10.3,243.0,NaN,NaN,NaN,1.91,NaN,NaN,21.799999,145.000000,NaN,2024-04-17,2024-05-19,m,B,89.0,0.0,NaN,33,NaN,NaN,NaT,NaT,0.0,0.0,4.0,55.0,19.0,7,2 | 4,0,NaN,0.0,No cancer,2024-05-19
5,0032c17f-9c2b-861a-1346-3a1d24b3e281,2024-04-24,7,36.299999,11.4,10.2,3.81,29.900000,31.400000,95.300003,14.8,0.3,1.9,0.9,7.1,10.0,260.0,20.0,31.0,NaN,1.94,NaN,4.710,NaN,147.199997,77.0,2024-04-17,2024-05-19,m,B,89.0,0.0,NaN,33,NaN,NaN,NaT,NaT,0.0,0.0,4.0,55.0,19.0,7,2 | 4,0,NaN,0.0,No cancer,2024-05-19
6,0032c17f-9c2b-861a-1346-3a1d24b3e281,2024-04-26,9,37.799999,11.7,11.8,3.94,29.700001,31.000000,95.900002,15.8,0.3,2.3,1.0,8.2,9.6,291.0,NaN,NaN,NaN,1.97,NaN,4.680,21.000000,145.000000,NaN,2024-04-17,2024-05-19,m,B,89.0,0.0,NaN,33,NaN,NaN,NaT,NaT,0.0,0.0,4.0,55.0,19.0,7,2 | 4,0,NaN,0.0,No cancer,2024-05-19
7,0032c17f-9c2b-861a-1346-3a1d24b3e281,2024-04-29,12,37.400002,11.8,11.0,3.84,30.700001,31.600000,97.400002,15.6,0.2,2.1,0.9,7.9,9.5,253.0,NaN,NaN,NaN,2.24,NaN,5.170,12.100000,153.800003,90.0,2024-04-17,2024-05-19,m,B,89.0,0.0,NaN,33,NaN,NaN,NaT,NaT,0.0,0.0,4.0,55.0,19.0,7,2 | 4,0,NaN,0.0,No cancer,2024-05-19
8,0032c17f-9c2b-861a-1346-3a1d24b3e281,2024-04-30,13,0.100000,0.0,1.1,0.01,0.000000,0.000000,100.000000,NaN,0.0,0.5,0.2,0.5,NaN,1.0,NaN,NaN,NaN,2.23,375.0,4.860,NaN,153.000000,98.0,2024-04-17,2024-05-19,m,B,89.0,0.0,NaN,33,NaN,NaN,NaT,NaT,0.0,0.0,4.0,55.0,19.0,7,2 | 4,0,NaN,0.0,No cancer,2024-05-19
9,0032c17f-9c2b-861a-1346-3a1d24b3e281,2024-05-01,14,38.200001,12.7,10.7,3.95,32.200001,33.200001,96.699997,15.9,0.1,1.8,0.9,7.9,9.8,228.0,NaN,NaN,NaN,2.24,NaN,

In [14]:
# Blood test columns (exclude structural/covariate columns)
blood_test_cols = [c for c in df_longitudinal.columns if c not in structural_cols]
print(blood_test_cols)

# Rows where ALL blood test columns are missing
mask_all_nan_blood = df_longitudinal[blood_test_cols].isna().all(axis=1)
rows_all_nan_blood = df_longitudinal[mask_all_nan_blood]

print(f"Blood test columns:                    {len(blood_test_cols)}")
print(f"Rows with all blood tests = NaN:       {mask_all_nan_blood.sum()}")

if mask_all_nan_blood.any():
    print("\nSample rows found:")
    display(rows_all_nan_blood[['patient_id', 'analysis_date', 'day'] + available_covariates].head(20))
else:
    print("No rows with all blood tests missing.")


['EMOCROMO - Ematocrito (%)', 'EMOCROMO - Emoglobina (g/dL)', 'EMOCROMO - Globuli bianchi (10^9/L)', 'EMOCROMO - Globuli rossi (10^12/L)', 'EMOCROMO - MCH (Emoglobina Corpuscolare Media) (pg/Cell)', 'EMOCROMO - MCHC (Conc.Emoglobinica Corpusc.Media) (g Hb /dL)', 'EMOCROMO - MCV (Volume Globulare Medio) (fL)', 'EMOCROMO - RDW (Ind.Distribuzione Eritrocitaria) (CV %)', 'FORMULA  - Eosinofili (10^9/L)', 'FORMULA  - Linfociti (10^9/L)', 'FORMULA  - Monociti (10^9/L)', 'FORMULA  - Neutrofili (10^9/L)', 'PIASTRINE - MPV (Volume Piastrinico Medio) (fL)', 'PIASTRINE - Piastrine (10^9/L)', 'S-ALANINA AMINOTRANSFERASI (U/L)', 'S-ASPARTATO AMINOTRANSFERASI (U/L)', 'S-BILIRUBINA TOTALE (mg/dL)', 'S-CREATININA (mg/dL)', 'S-LATTICODEIDROGENASI (U/L)', 'S-POTASSIO (mmol/L)', 'S-PROTEINA C REATTIVA (mg/L)', 'S-SODIO (mmol/L)', 'S-UREA (mg/dL)']
Blood test columns:                    23
Rows with all blood tests = NaN:       0
No rows with all blood tests missing.


In [15]:
# Remove rows from df_longitudinal where all blood test columns are NaN
n_rows_to_remove = int(mask_all_nan_blood.sum())

df_longitudinal = df_longitudinal.loc[~mask_all_nan_blood].reset_index(drop=True)

print(f"Rows removed: {n_rows_to_remove}")
print(f"New shape:    {df_longitudinal.shape}")


Rows removed: 0
New shape:    (14760, 50)


In [16]:
# ============================================================
#  10  MICE IMPUTATION OF ADMISSION COVARIATES
# ============================================================
# Strategy:
#   1. Extract one row per patient (covariates are cross-sectional)
#   2. Apply MICE (IterativeImputer) on that patient-level table
#   3. Map imputed values back to every row of df_longitudinal
#
# NOTE: dateOfDeath / startDate / endDate / ward_startDate / ER_last_day are
#       EXCLUDED from MICE because they are date / outcome columns we don't
#       want to impute. They are passed through unchanged.
# ============================================================

from sklearn.experimental import enable_iterative_imputer   # noqa: F401
from sklearn.impute import IterativeImputer
from sklearn.ensemble import RandomForestRegressor
import numpy as np

# --- CONFIGURATION ---
MICE_MAX_ITER     = 10
MICE_N_NEAREST    = 5
MICE_RANDOM_STATE = 42

COVARIATE_CAT_COLS = [
    'hospitalWard',
    'sex',
    'admReason',
    'dischargeMode',
    'dischargedAlive',
    'solidTumor',
    'solidTumor_cat',
]

# Date / outcome columns excluded from MICE (passed through unchanged)
COVARIATE_PASSTHROUGH_COLS = [
    'startDate',
    'endDate',
    'ward_startDate',
    'ER_last_day',
    'dateOfDeath',
]

# ── Step 1: build patient-level covariate table ──────────────────────────────
cov_patient = (
    df_longitudinal[['patient_id'] + available_covariates]
    .drop_duplicates(subset='patient_id')
    .set_index('patient_id')
    .copy()
)

print(f"  Patient-level covariate table: {cov_patient.shape}")
print(f"  Covariates: {available_covariates}")

cols_to_impute           = [c for c in available_covariates if c not in COVARIATE_PASSTHROUGH_COLS]
cols_passthrough_present = [c for c in COVARIATE_PASSTHROUGH_COLS if c in available_covariates]

print(f"\n  Columns going into MICE:   {cols_to_impute}")
print(f"  Columns passed through:    {cols_passthrough_present}")

cov_to_impute   = cov_patient[cols_to_impute].copy()
cov_passthrough = cov_patient[cols_passthrough_present].copy()

nan_before     = cov_to_impute.isnull().sum()
nan_before_pct = cov_to_impute.isnull().mean() * 100
print("\n  NaN per covariate BEFORE imputation:")
for col in cols_to_impute:
    print(f"   {col:<40s}  {int(nan_before[col]):>4d}  ({nan_before_pct[col]:.1f}%)")

# ── Step 2: encode categoricals, run MICE, decode ────────────────────────────
categorical_cols = [c for c in COVARIATE_CAT_COLS if c in cols_to_impute]
numeric_cols     = [c for c in cols_to_impute if c not in categorical_cols]

cov_encoded = cov_to_impute.copy()
label_encoders = {}
for col in categorical_cols:
    cov_encoded[col] = cov_encoded[col].astype('category')
    label_encoders[col] = dict(enumerate(cov_encoded[col].cat.categories))
    cov_encoded[col] = cov_encoded[col].cat.codes.replace(-1, np.nan)

imputer = IterativeImputer(
    estimator=RandomForestRegressor(n_estimators=100, random_state=MICE_RANDOM_STATE),
    max_iter=MICE_MAX_ITER,
    n_nearest_features=MICE_N_NEAREST,
    random_state=MICE_RANDOM_STATE,
    initial_strategy='most_frequent',
)
imputed_array = imputer.fit_transform(cov_encoded.values)
cov_imputed   = pd.DataFrame(
    imputed_array, index=cov_to_impute.index, columns=cov_to_impute.columns
)

for col in categorical_cols:
    if not label_encoders[col]:
        continue
    codes = cov_imputed[col].round().astype(int).clip(0, max(label_encoders[col].keys()))
    cov_imputed[col] = codes.map(label_encoders[col])

for col in numeric_cols:
    if (cov_to_impute[col].dropna() % 1 == 0).all():
        cov_imputed[col] = cov_imputed[col].round().astype(float)

# Re-attach pass-through columns
cov_imputed = cov_imputed.join(cov_passthrough, how='left')

print("\n  MICE imputation completed.")
nan_after = cov_imputed[cols_to_impute].isnull().sum().sum()
if nan_after == 0:
    print("  OK  No NaN remaining in imputed covariates.")
else:
    print(f"  WARNING: {nan_after} NaN still present after imputation.")

# ── Step 3: propagate imputed values back to df_longitudinal ─────────────────
df_longitudinal = df_longitudinal.drop(columns=available_covariates)
df_longitudinal = df_longitudinal.merge(
    cov_imputed.reset_index(),
    on='patient_id',
    how='left',
)

structural_cols = ['patient_id', 'analysis_date', 'day']
test_cols_final = [
    c for c in df_longitudinal.columns
    if c not in structural_cols and c not in available_covariates
]
df_longitudinal = df_longitudinal[structural_cols + test_cols_final + available_covariates]

print(f"\n  df_longitudinal after covariate imputation: {df_longitudinal.shape}")
print(f"   Patients:       {df_longitudinal['patient_id'].nunique()}")
print(f"   Covariate NaN:  {df_longitudinal[cols_to_impute].isnull().sum().sum()} "
      f"(in imputable cols)")


  Patient-level covariate table: (1203, 24)
  Covariates: ['startDate', 'endDate', 'sex', 'hospitalWard', 'age', 'pksYear', 'alcYears', 'LOS_days', 'daysInER', 'daysInER_capped', 'ER_last_day', 'ward_startDate', 'admLast12m', 'admLast6m', 'admECOG', 'admBarthelScore', 'admBradenScore', 'cirsTotalScore', 'admReason', 'dischargedAlive', 'dischargeMode', 'solidTumor', 'solidTumor_cat', 'dateOfDeath']

  Columns going into MICE:   ['sex', 'hospitalWard', 'age', 'pksYear', 'alcYears', 'LOS_days', 'daysInER', 'daysInER_capped', 'admLast12m', 'admLast6m', 'admECOG', 'admBarthelScore', 'admBradenScore', 'cirsTotalScore', 'admReason', 'dischargedAlive', 'dischargeMode', 'solidTumor', 'solidTumor_cat']
  Columns passed through:    ['startDate', 'endDate', 'ward_startDate', 'ER_last_day', 'dateOfDeath']

  NaN per covariate BEFORE imputation:
   sex                                         35  (2.9%)
   hospitalWard                                 0  (0.0%)
   age                                  


  MICE imputation completed.
  OK  No NaN remaining in imputed covariates.

  df_longitudinal after covariate imputation: (14760, 50)
   Patients:       1203
   Covariate NaN:  0 (in imputable cols)


  Patient-level covariate table: (1203, 24)
  Covariates: ['startDate', 'endDate', 'sex', 'hospitalWard', 'age', 'pksYear', 'alcYears', 'LOS_days', 'daysInER', 'daysInER_capped', 'ER_last_day', 'ward_startDate', 'admLast12m', 'admLast6m', 'admECOG', 'admBarthelScore', 'admBradenScore', 'cirsTotalScore', 'admReason', 'dischargedAlive', 'dischargeMode', 'solidTumor', 'solidTumor_cat', 'dateOfDeath']

  Columns going into MICE:   ['sex', 'hospitalWard', 'age', 'pksYear', 'alcYears', 'LOS_days', 'daysInER', 'daysInER_capped', 'admLast12m', 'admLast6m', 'admECOG', 'admBarthelScore', 'admBradenScore', 'cirsTotalScore', 'admReason', 'dischargedAlive', 'dischargeMode', 'solidTumor', 'solidTumor_cat']
  Columns passed through:    ['startDate', 'endDate', 'ward_startDate', 'ER_last_day', 'dateOfDeath']

  NaN per covariate BEFORE imputation:
   sex                                         35  (2.9%)
   hospitalWard                                 0  (0.0%)
   age                                  

In [17]:
# Sanity check: if admLast6m > admLast12m, set admLast12m = admLast6m
# (the 12-month count cannot logically be smaller than the 6-month count)

if 'admLast6m' in df_longitudinal.columns and 'admLast12m' in df_longitudinal.columns:
    mask = (
        df_longitudinal['admLast6m'].notna() &
        df_longitudinal['admLast12m'].notna() &
        (df_longitudinal['admLast6m'] > df_longitudinal['admLast12m'])
    )

    n_issues = int(mask.sum())
    df_longitudinal.loc[mask, 'admLast12m'] = df_longitudinal.loc[mask, 'admLast6m']

    print(f"Numero di correzioni effettuate: {n_issues}")
else:
    print("Colonne admLast6m / admLast12m non presenti — nessun controllo eseguito.")


Numero di correzioni effettuate: 74


In [18]:
# ITA → ENG column mapping (covariate names match 01_clinical.parquet)
column_mapping = {
    'patient_id': 'patient_id',
    'analysis_date': 'analysis_date',
    'day': 'day',

    # --- Blood tests (23 whitelisted analyses) ---
    'EMOCROMO - Ematocrito (%)': 'HCT (%)',
    'EMOCROMO - Emoglobina (g/dL)': 'HGB (g/dL)',
    'EMOCROMO - Globuli bianchi (10^9/L)': 'WBC (10^9/L)',
    'EMOCROMO - Globuli rossi (10^12/L)': 'RBC (10^12/L)',
    'EMOCROMO - MCH (Emoglobina Corpuscolare Media) (pg/Cell)': 'MCH (pg)',
    'EMOCROMO - MCHC (Conc.Emoglobinica Corpusc.Media) (g Hb /dL)': 'MCHC (g/dL)',
    'EMOCROMO - MCV (Volume Globulare Medio) (fL)': 'MCV (fL)',
    'EMOCROMO - RDW (Ind.Distribuzione Eritrocitaria) (CV %)': 'RDW (%)',

    'FORMULA  - Linfociti (10^9/L)': 'LYM (10^9/L)',
    'FORMULA  - Eosinofili (10^9/L)': 'EOS (10^9/L)',
    'FORMULA  - Monociti (10^9/L)': 'MONO (10^9/L)',
    'FORMULA  - Neutrofili (10^9/L)': 'NEU (10^9/L)',

    'PIASTRINE - MPV (Volume Piastrinico Medio) (fL)': 'MPV (fL)',
    'PIASTRINE - Piastrine (10^9/L)': 'PLT (10^9/L)',

    'S-ALANINA AMINOTRANSFERASI (U/L)': 'ALT (U/L)',
    'S-ASPARTATO AMINOTRANSFERASI (U/L)': 'AST (U/L)',
    'S-BILIRUBINA TOTALE (mg/dL)': 'TBIL (mg/dL)',
    'S-CREATININA (mg/dL)': 'CREA (mg/dL)',
    'S-LATTICODEIDROGENASI (U/L)': 'LDH (U/L)',
    'S-POTASSIO (mmol/L)': 'K (mmol/L)',
    'S-PROTEINA C REATTIVA (mg/L)': 'CRP (mg/L)',
    'S-SODIO (mmol/L)': 'Na (mmol/L)',
    'S-UREA (mg/dL)': 'UREA (mg/dL)',

    # --- Admission covariates (from 01_clinical.parquet) ---
    'startDate': 'admission_start_date',
    'endDate': 'admission_end_date',
    'sex': 'sex',
    'hospitalWard': 'hospital_ward',
    'age': 'age',
    'pksYear': 'pack_years_smoking',
    'alcYears': 'alcohol_years',
    'LOS_days': 'length_of_stay_days',
    'daysInER': 'days_in_ER',
    'daysInER_capped': 'days_in_ER_capped',
    'ER_last_day': 'ER_last_day',
    'ward_startDate': 'ward_start_date',
    'admLast12m': 'admissions_last_12m',
    'admLast6m': 'admissions_last_6m',
    'admECOG': 'ECOG',
    'admBarthelScore': 'Barthel_index',
    'admBradenScore': 'Braden_score',
    'cirsTotalScore': 'CIRS_total_score',
    'admReason': 'admission_reason',
    'dischargedAlive': 'discharged_alive',
    'dischargeMode': 'discharge_mode',
    'solidTumor': 'solid_tumor',
    'solidTumor_cat': 'solid_tumor_category',

    # --- Death (from new_df_episodekey_deathdate.csv) ---
    'dateOfDeath': 'date_of_death',
}

df_longitudinal = df_longitudinal.rename(columns=column_mapping)

missing_cols = set(df_longitudinal.columns) - set(column_mapping.values())
print("Columns not mapped:", missing_cols)


Columns not mapped: set()


In [19]:
# ============================================================
#  11  SAVE DATASET TO CSV
# ============================================================

from pathlib import Path

OUTPUT_DIR = Path("/home/azureuser/cloudfiles/code/Users/cecilia.marco/UseCase-Code/data-science/projects/longitudinal_HMM/dataset creation")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_FILE = OUTPUT_DIR / "df_longitudinal.csv"

df_longitudinal.to_csv(OUTPUT_FILE, index=False)

print(f"✅ Dataset saved to: {OUTPUT_FILE}")
print(f"   Rows:    {len(df_longitudinal):,}")
print(f"   Columns: {len(df_longitudinal.columns)}")
print(f"   Size:    {OUTPUT_FILE.stat().st_size / (1024**2):.2f} MB")


✅ Dataset saved to: /home/azureuser/cloudfiles/code/Users/cecilia.marco/UseCase-Code/data-science/projects/longitudinal_HMM/dataset creation/df_longitudinal.csv
   Rows:    14,760
   Columns: 50
   Size:    6.25 MB


In [20]:
# ============================================================
#  12  SAVE CLINICAL REFERENCE RANGES (for the training notebook)
# ============================================================
# Remap the reference-range keys from the pre-rename wide column name to
# the SHORT display names used downstream (same column_mapping as the
# main dataset), then persist next to df_longitudinal.csv. The HMM
# training notebook reads this file to draw the normal-range bands
# (sections 3.5 / 7.1) and the 'Normal Value' column (section 8).

ref_ranges = ref_ranges_longname.copy()
ref_ranges['biomarker'] = ref_ranges['test_col_name'].map(column_mapping)

unmapped = ref_ranges.loc[ref_ranges['biomarker'].isna(), 'test_col_name'].tolist()
if unmapped:
    print("  WARNING: these reference-range tests are not in column_mapping "
          "(skipped — check unit strings match section 16):")
    for u in unmapped:
        print(f"   - {u}")

ref_ranges = (
    ref_ranges
    .dropna(subset=['biomarker'])
    .loc[:, ['biomarker', 'lln', 'uln', 'n_distinct_ranges', 'n_ranges_kept',
             'n_measurements', 'n_measurements_kept', 'n_unparsed']]
    .sort_values('biomarker')
    .reset_index(drop=True)
)

REF_OUTPUT_FILE = OUTPUT_DIR / "biomarker_reference_ranges.csv"
ref_ranges.to_csv(REF_OUTPUT_FILE, index=False)

print(f"✅ Reference ranges saved to: {REF_OUTPUT_FILE}")
print(f"   Tests: {len(ref_ranges)}")
print()
print(ref_ranges.to_string(index=False))


✅ Reference ranges saved to: /home/azureuser/cloudfiles/code/Users/cecilia.marco/UseCase-Code/data-science/projects/longitudinal_HMM/dataset creation/biomarker_reference_ranges.csv
   Tests: 23

    biomarker        lln        uln  n_distinct_ranges  n_ranges_kept  n_measurements  n_measurements_kept  n_unparsed
    ALT (U/L)   6.000000  51.738636                  2              2           24816                24816           0
    AST (U/L)   5.000000  34.189851                  2              2           24909                24909           0
 CREA (mg/dL)   0.500000   1.190408                  4              2           35537                35534           0
   CRP (mg/L)   0.000000   6.000000                  1              1           28156                28156           0
 EOS (10^9/L)   0.000000   0.450000                  1              1           38457                38457           0
      HCT (%)  40.008734  50.008734                  2              2           42247      